In [54]:
import re
from math import ceil
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats
from iminuit import Minuit

# ---- Paper/LaTeX-ish defaults (no TeX install needed) ----
mpl.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.size": 11,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "legend.frameon": False,
})


def autoscale_fonts(fig, base=10, ref_width=6.4):
    scale = fig.get_figwidth() / ref_width
    fs = base * scale

    for ax in fig.axes:
        ax.title.set_fontsize(fs * 1.1)
        ax.xaxis.label.set_fontsize(fs)
        ax.yaxis.label.set_fontsize(fs)
        ax.tick_params(labelsize=fs * 0.9)

        leg = ax.get_legend()
        if leg is not None:
            for t in leg.get_texts():
                t.set_fontsize(fs * 0.9)

    if fig._suptitle is not None:
        fig._suptitle.set_fontsize(fs * 1.2)

    return fs


def prettify(a):
    a.minorticks_on()
    a.grid(True, which="major", linewidth=0.6, alpha=0.35)
    a.grid(True, which="minor", linewidth=0.4, alpha=0.20)
    a.tick_params(which="both", top=False, right=False)

In [55]:
# ---- Masses (MeV) ----
mA = 20484.845566
ma = 6535.365833
mb = 2809.432119
mB = 24202.632149


def parse_file_info(path):
    match = re.match(
        r"Ex(?P<ex>\d+p\d+)MeV_(?P<lo>\d+p\d+)to(?P<hi>\d+p\d+)deg\.dat",
        path.name,
    )
    if not match:
        return None
    ex = float(match.group("ex").replace("p", "."))
    lo = float(match.group("lo").replace("p", "."))
    hi = float(match.group("hi").replace("p", "."))
    return ex, lo, hi


def load_gate_data(path):
    data = np.loadtxt(path, dtype=float, delimiter=",")
    e_t = data[:, 8]
    thetalab_t = data[:, 9]
    ebeam = data[:, -1]
    return ebeam, e_t, thetalab_t


def compute_excitation(ebeam, e_t, thetalab_t):
    val1 = mA**2 + ma**2 + mb**2 + 2 * mA * ma + 2 * ma * ebeam
    val2 = -2 * (mb + e_t) * (mA + ebeam + ma)
    val3 = (
        2
        * np.cos(np.deg2rad(thetalab_t))
        * np.sqrt(ebeam**2 + 2 * mA * ebeam)
        * np.sqrt(e_t**2 + 2 * mb * e_t)
    )
    return np.sqrt(val1 + val2 + val3) - mB


def fit_excitation(exc):
    if exc.size < 20:
        return None
    hi = np.mean(exc) + 2.0 * np.std(exc)
    lo = np.mean(exc) - 2.0 * np.std(exc)
    exc = exc[(exc > lo) & (exc < hi)]
    if exc.size < 20:
        return None

    def nll(sigma, e_mean, f):
        if sigma <= 0 or f <= 0 or f >= 1:
            return 1e50
        z = stats.norm.cdf(hi, loc=e_mean, scale=sigma) - stats.norm.cdf(
            lo, loc=e_mean, scale=sigma
        )
        if (not np.isfinite(z)) or (z <= 0):
            return 1e50
        log_s = stats.norm.logpdf(exc, loc=e_mean, scale=sigma) - np.log(z)
        log_b = -np.log(hi - lo)
        return -np.sum(np.logaddexp(np.log(f) + log_s, np.log(1 - f) + log_b))

    sigma_guess = np.std(exc)
    e_guess = np.mean(exc)
    f_guess = 0.8

    m = Minuit(nll, sigma=sigma_guess, e_mean=e_guess, f=f_guess)
    m.limits["sigma"] = (1e-9, None)
    m.errordef = Minuit.LIKELIHOOD
    m.migrad()
    m.hesse()

    sigma_hat = m.values["sigma"]
    sigma_err = m.errors["sigma"]
    e_hat = m.values["e_mean"]
    e_err = m.errors["e_mean"]
    f_hat = m.values["f"]
    f_err = m.errors["f"]

    z = stats.norm.cdf(hi, loc=e_hat, scale=sigma_hat) - stats.norm.cdf(
        lo, loc=e_hat, scale=sigma_hat
    )

    def model_cdf(t):
        t = np.asarray(t, dtype=float)
        out = np.empty_like(t)

        out[t <= lo] = 0.0
        out[t >= hi] = 1.0

        mask = (t > lo) & (t < hi)
        tt = t[mask]

        fs = (
            stats.norm.cdf(tt, loc=e_hat, scale=sigma_hat)
            - stats.norm.cdf(lo, loc=e_hat, scale=sigma_hat)
        ) / z
        fb = (tt - lo) / (hi - lo)

        out[mask] = f_hat * fs + (1.0 - f_hat) * fb
        return out

    d_stat, p_value = stats.kstest(exc, model_cdf)
    return {
        "exc": exc,
        "lo": lo,
        "hi": hi,
        "sigma_hat": sigma_hat,
        "sigma_err": sigma_err,
        "e_hat": e_hat,
        "e_err": e_err,
        "f_hat": f_hat,
        "f_err": f_err,
        "d_stat": d_stat,
        "p_value": p_value,
    }


def main():
    paths = sorted(
        Path(".").glob("Ex*p*MeV_*deg.dat"),
        key=lambda p: parse_file_info(p) or (np.inf, np.inf, np.inf),
    )
    paths = [p for p in paths if parse_file_info(p) is not None]

    if not paths:
        raise FileNotFoundError("No Ex*p*MeV_*deg.dat files found in this folder.")

    ncols = 4
    nrows = ceil(len(paths) / ncols)
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(ncols * 5, nrows * 5), constrained_layout=True
    )
    axes = np.atleast_1d(axes).ravel()

    for ax, path in zip(axes, paths):
        info = parse_file_info(path)
        ex, lo_deg, hi_deg = info
        ebeam_i, e_t, thetalab_t = load_gate_data(path)
        exc = compute_excitation(ebeam_i, e_t, thetalab_t)
        result = fit_excitation(exc)

        if result is None:
            ax.text(0.5, 0.5, "Not enough points for fit", ha="center", va="center")
            ax.set_axis_off()
            continue

        exc = result["exc"]
        
        sigma_hat = result["sigma_hat"]
        sigma_err = result["sigma_err"]
        e_hat = result["e_hat"]
        e_err = result["e_err"]
        f_hat = result["f_hat"]
        f_err = result["f_err"]
        lo = result["lo"]
        hi = result["hi"]
        p_value = result["p_value"]

        counts, edges, _ = ax.hist(exc, bins=200, histtype="step", linewidth=0.7)
        bin_width = edges[1] - edges[0]

        e = np.linspace(min(exc), max(exc), 1000)
        z = stats.norm.cdf(hi, loc=e_hat, scale=sigma_hat) - stats.norm.cdf(
            lo, loc=e_hat, scale=sigma_hat
        )
        signal_density = stats.norm.pdf(e, loc=e_hat, scale=sigma_hat) / z
        bkg_density = 1.0 / (hi - lo)
        expected_counts = exc.size * bin_width * (
            f_hat * signal_density + (1 - f_hat) * bkg_density
        )
        ax.plot(e, expected_counts, color="#FE019A", linewidth=1.0, label="Fit")

        ax.set_title(
            rf"$E_x={ex:.3f}$ MeV, $\theta$=[{lo_deg:.2f}, {hi_deg:.2f}]$^\circ$"
        )
        ax.set_xlabel("Excitation energy (MeV)")
        ax.set_ylabel("Counts")
        ax.text(
            0.02,
            0.95,
            f"$E_x = {e_hat:.3f} \pm {e_err:.1e} MeV$ \n$\sigma = {sigma_hat:.3f} \pm {sigma_err:.1e} MeV$\n$p = {p_value:.3g}$",
            transform=ax.transAxes,
            va="top",
        )
        prettify(ax)

    for ax in axes[len(paths):]:
        ax.set_axis_off()
       
        autoscale_fonts(fig, base=4)
        plt.tight_layout()
        
        

    
    # plt.savefig("MultipleFits.png")
    


if __name__ == "__main__":
    main()

<>:181: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<>:181: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:181: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<>:181: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<>:181: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:181: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
/var/folders/jr/p39hf4vd6lg896bktf3v_y800000gn/T/ipykernel_22978/33440

In [56]:
ex = np.loadtxt("Exc.csv", delimiter=",")
print(ex[:, 1])

[ 4.33      4.9       5.47      6.87      7.37      8.21      9.38
 10.696787 10.805787 10.817787 10.825787 10.948787 11.083787 11.320787
 11.336787 11.827787 11.894787 11.910787 11.951787 12.270787 12.344787]
